In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import time
import os

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from codecarbon import EmissionsTracker


In [ ]:
BASE_RESULTS_DIR = "/content/drive/MyDrive/green-ai-cnn-carbon/results/cifar10"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)


In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)


In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input

x_train_e = preprocess_input(x_train * 255.0)
x_test_e  = preprocess_input(x_test * 255.0)


Below are the notebook codes for defining each model which have be run seperately for per epoch training

In [ ]:
def get_lenet5():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(6, (5,5), activation='relu', input_shape=(32,32,3)),
        tf.keras.layers.AveragePooling2D(),
        tf.keras.layers.Conv2D(16, (5,5), activation='relu'),
        tf.keras.layers.AveragePooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(120, activation='relu'),
        tf.keras.layers.Dense(84, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
def get_vgg16():
    base = tf.keras.applications.VGG16(
        weights=None,
        include_top=False,
        input_shape=(32,32,3)
    )

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    output = tf.keras.layers.Dense(10, activation="softmax")(x)

    model = tf.keras.Model(base.input, output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
def get_resnet18():
    inputs = tf.keras.Input(shape=(32, 32, 3))
    
    # CIFAR stem (small + fast)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    
    # Stage 1
    for _ in range(2):
        x = resnet_block(x, 64, stride=1)
    
    # Stage 2
    x = resnet_block(x, 128, stride=2)
    x = resnet_block(x, 128, stride=1)
    
    # Stage 3
    x = resnet_block(x, 256, stride=2)
    x = resnet_block(x, 256, stride=1)
    
    # Classification head
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
    
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    return model

In [ ]:
def get_resnet50():
    base = tf.keras.applications.ResNet50(
        weights=None,
        include_top=False,
        input_shape=(32,32,3)
    )

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    output = tf.keras.layers.Dense(10, activation="softmax")(x)

    model = tf.keras.Model(base.input, output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

x_train_m = preprocess_input(x_train * 255.0)
x_test_m  = preprocess_input(x_test * 255.0)

def get_mobilenetv2():
    base = tf.keras.applications.MobileNetV2(
        weights=None,
        include_top=False,
        input_shape=(32,32,3)
    )

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    output = tf.keras.layers.Dense(10, activation="softmax")(x)

    model = tf.keras.Model(base.input, output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input

x_train_e = preprocess_input(x_train * 255.0)
x_test_e  = preprocess_input(x_test * 255.0)

def get_efficientnetb0():
    base = tf.keras.applications.EfficientNetB0(
        weights=None,
        include_top=False,
        input_shape=(32,32,3)
    )

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    output = tf.keras.layers.Dense(10, activation="softmax")(x)

    model = tf.keras.Model(base.input, output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
MODEL_NAME = ""
model = get_MODEL_NAME()
model.compile(
    optimizer=tf.keras.optimizers.Adam(), #enter learning_rate
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64

tracker = EmissionsTracker(
    project_name=f"carbon_budget_{MODEL_NAME}",
    output_dir="/content/drive/MyDrive/green-ai-cnn-carbon/tracking",
    log_level="error"
)

epoch_logs = []
tracker.start()
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")

    model.fit(
        x_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=1,
        verbose=1
    )

    loss, acc = model.evaluate(x_test, y_test, verbose=0)

    emissions_kg = tracker.flush()
    emissions_g = emissions_kg * 1000

    elapsed_min = (time.time() - start_time) / 60

    epoch_logs.append({
        "dataset": "CIFAR-10",
        "model": MODEL_NAME,
        "epoch": epoch + 1,
        "val_accuracy": acc,
        "cumulative_co2_g": emissions_g,
        "elapsed_time_min": elapsed_min
    })

tracker.stop()

# 🔽 AUTO-SAVE IMMEDIATELY AFTER TRAINING
df = pd.DataFrame(epoch_logs)
csv_path = f"{BASE_RESULTS_DIR}/epochwise_carbon_{MODEL_NAME}.csv"
df.to_csv(csv_path, index=False)

print(f"Saved → {csv_path}")
df.head()


In [ ]:
df = pd.DataFrame(epoch_logs)

csv_path = f"{BASE_RESULTS_DIR}/epochwise_carbon_{MODEL_NAME}.csv"
df.to_csv(csv_path, index=False)

print(f"Saved → {csv_path}")
df.head()


In [ ]:
CARBON_BUDGETS = [1, 5, 10]  # grams CO₂
results = []

for model in df_all["model"].unique():
    df_m = df_all[df_all["model"] == model]

    for B in CARBON_BUDGETS:
        df_budget = df_m[df_m["cumulative_co2_g"] <= B]

        acc_B = df_budget["val_accuracy"].max() if not df_budget.empty else 0

        results.append({
            "model": model,
            "carbon_budget_g": B,
            "carbon_budgeted_accuracy": acc_B
        })

df_budgeted = pd.DataFrame(results)
df_budgeted